In [1]:
import duckdb

con = duckdb.connect('../superstore.db')

con.execute("""
    CREATE OR REPLACE TABLE stg AS
    SELECT * FROM read_csv_auto('../data/superstore.csv')
""")

In [3]:
con.execute("SELECT COUNT(*) AS rows FROM stg").df()

,rows
0,5901


In [5]:
con.execute("""
    SELECT COUNT(*) AS total_rows, COUNT(DISTINCT "Order ID") AS distinct_orders
    FROM stg
""").df()

,total_rows,distinct_orders
0,5901,3003


In [7]:
con.execute("""
    SELECT "Order ID", "Product Name", "Category", "Sales"
    FROM stg
    WHERE "Order ID" = (
        SELECT "Order ID" FROM stg GROUP BY "Order ID" HAVING COUNT(*) > 1 LIMIT 1
    )
""").df()

,Order ID,Product Name,Category,Sales
0,CA-2019-168046,"GBC Standard Recycled Report Covers, Clear Pla...",Office Supplies,43.120
1,CA-2019-168046,Chromcraft Round Conference Tables,Furniture,313.722
2,CA-2019-168046,Tenex B1-RE Series Chair Mats for Low Pile Car...,Furniture,45.980
3,CA-2019-168046,"SAFCO Commercial Wire Shelving, 72h",Office Supplies,428.680


In [9]:
con.execute("""
    CREATE OR REPLACE TABLE products AS
    SELECT
        "Product ID"   AS product_id,
        MAX("Category")     AS category,
        MAX("Sub-Category") AS sub_category,
        MAX("Product Name") AS product_name
    FROM stg
    GROUP BY "Product ID"
""")

In [13]:
con.execute("""
    CREATE OR REPLACE TABLE orders AS
    SELECT
        "Order ID"    AS order_id,
        MAX("Order Date")   AS order_date,
        MAX("Ship Date")    AS ship_date,
        MAX("Ship Mode")    AS ship_mode,
        MAX("Customer ID")  AS customer_id,
        MAX("Segment")      AS segment,
        MAX("Region")       AS region,
        MAX("State")        AS state
    FROM stg
    GROUP BY "Order ID"
""")

In [15]:
con.execute("""
    CREATE OR REPLACE TABLE order_lines AS
    SELECT
        "Row ID"     AS line_id,
        "Order ID"   AS order_id,
        "Product ID" AS product_id,
        "Sales"      AS sales,
        "Quantity"   AS quantity,
        "Discount"   AS discount,
        "Profit"     AS profit
    FROM stg
""")

BinderException: Binder Error: Referenced column "Row ID" not found in FROM clause!
Candidate bindings: "Row ID+O6G3A1:R6", "Product ID", "Order ID", "Ship Date", "Order Date"

LINE 4:         "Row ID"     AS line_id,
                ^

In [17]:
con.execute("""
    CREATE OR REPLACE TABLE order_lines AS
    SELECT
        "Row ID"     AS line_id,
        "Order ID"   AS order_id,
        "Product ID" AS product_id,
        "Sales"      AS sales,
        "Quantity"   AS quantity,
        "Discount"   AS discount,
        "Profit"     AS profit
    FROM stg
""")

BinderException: Binder Error: Referenced column "Row ID" not found in FROM clause!
Candidate bindings: "Row ID+O6G3A1:R6", "Product ID", "Order ID", "Ship Date", "Order Date"

LINE 4:         "Row ID"     AS line_id,
                ^

In [19]:
con.execute("DESCRIBE stg").df()

,column_name,column_type,null,key,default,extra
0,Row ID+O6G3A1:R6,BIGINT,YES,None,None,None
1,Order ID,VARCHAR,YES,None,None,None
2,Order Date,DATE,YES,None,None,None
3,Ship Date,DATE,YES,None,None,None
4,Ship Mode,VARCHAR,YES,None,None,None
5,Customer ID,VARCHAR,YES,None,None,None
6,Customer Name,VARCHAR,YES,None,None,None
7,Segment,VARCHAR,YES,None,None,None
8,Country,VARCHAR,YES,None,None,None
9,City,VARCHAR,YES,None,None,None


In [23]:
con.execute("""
    ALTER TABLE stg RENAME COLUMN "Row ID+O6G3A1:R6" TO "Row ID"
""")

In [25]:
con.execute("""
    CREATE OR REPLACE TABLE order_lines AS
    SELECT
        "Row ID"     AS line_id,
        "Order ID"   AS order_id,
        "Product ID" AS product_id,
        "Sales"      AS sales,
        "Quantity"   AS quantity,
        "Discount"   AS discount,
        "Profit"     AS profit
    FROM stg
""")

BinderException: Binder Error: Column "Discount" referenced that exists in the SELECT clause - but this column cannot be referenced before it is defined

In [27]:
desc = con.execute("DESCRIBE stg").df()
desc['column_name'].value_counts()

column_name
Row ID           1
Product ID       1
ind1             1
Payment Mode     1
Returns          1
Profit           1
Quantity         1
Sales            1
Product Name     1
Sub-Category     1
Category         1
Region           1
Order ID         1
State            1
City             1
Country          1
Segment          1
Customer Name    1
Customer ID      1
Ship Mode        1
Ship Date        1
Order Date       1
ind2             1
Name: count, dtype: int64

In [29]:
print(desc['column_name'].tolist())

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Profit', 'Returns', 'Payment Mode', 'ind1', 'ind2']


In [31]:
con.execute("""
    CREATE OR REPLACE TABLE order_lines AS
    SELECT
        "Row ID"     AS line_id,
        "Order ID"   AS order_id,
        "Product ID" AS product_id,
        "Sales"      AS sales,
        "Quantity"   AS quantity,
        "Profit"     AS profit
    FROM stg
""")

In [33]:
con.execute("""
    SELECT
        (SELECT COUNT(*) FROM order_lines) AS line_count,
        (SELECT COUNT(*) FROM orders)      AS order_count,
        (SELECT COUNT(*) FROM products)    AS product_count
""").df()

,line_count,order_count,product_count
0,5901,3003,1755


In [35]:
con.execute("DESCRIBE orders").df()

,column_name,column_type,null,key,default,extra
0,order_id,VARCHAR,YES,None,None,None
1,order_date,DATE,YES,None,None,None
2,ship_date,DATE,YES,None,None,None
3,ship_mode,VARCHAR,YES,None,None,None
4,customer_id,VARCHAR,YES,None,None,None
5,segment,VARCHAR,YES,None,None,None
6,region,VARCHAR,YES,None,None,None
7,state,VARCHAR,YES,None,None,None


In [37]:
monthly = con.execute("""
    SELECT
        date_trunc('month', o.order_date) AS month,
        p.category,
        SUM(ol.sales)  AS total_sales,
        SUM(ol.profit) AS total_profit,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM order_lines ol
    JOIN orders o   ON ol.order_id = o.order_id
    JOIN products p ON ol.product_id = p.product_id
    GROUP BY date_trunc('month', o.order_date), p.category
    ORDER BY month, category
""").df()

monthly

,month,category,total_sales,total_profit,order_count
0,2019-01-01,Furniture,7696.6830,332.2275,18
1,2019-01-01,Office Supplies,5299.6820,1604.4712,32
2,2019-01-01,Technology,5620.0660,916.3914,16
3,2019-02-01,Furniture,3925.5510,377.0352,17
4,2019-02-01,Office Supplies,7794.3500,1330.0009,31
...,...,...,...,...,...
67,2020-11-01,Office Supplies,50028.3370,3609.1029,180
68,2020-11-01,Technology,41577.7730,5674.9371,79
69,2020-12-01,Furniture,40848.4668,1146.7548,84
70,2020-12-01,Office Supplies,98975.4720,1773.7084,172


In [39]:
len(monthly)

72

In [41]:
monthly.to_csv('../data/monthly_sales_by_category.csv', index=False)